In [1]:
# ===== Cell 1: Setup =====
!pip install -q kagglehub

import os
import shutil
import glob
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import kagglehub

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

DATA_ROOT = "data"
os.makedirs(DATA_ROOT, exist_ok=True)

print("Setup complete. DATA_ROOT =", DATA_ROOT)


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\profe\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



Setup complete. DATA_ROOT = data


In [2]:
# ===== Cell 2: Download datasets via kagglehub =====

# 1) Skin cancer malignant vs benign (fanconic)
path_skin_simple = kagglehub.dataset_download("fanconic/skin-cancer-malignant-vs-benign")
print("SkinCancer kagglehub path:", path_skin_simple)
print("Contents:", os.listdir(path_skin_simple))

# 2) Chest X-ray pneumonia
path_chest = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
print("ChestXray kagglehub path:", path_chest)
print("Contents:", os.listdir(path_chest))

# 3) HAM10000
path_ham = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
print("HAM10000 kagglehub path:", path_ham)
print("Contents:", os.listdir(path_ham))

100%|██████████| 325M/325M [05:09<00:00, 1.10MB/s] 

Extracting files...


SkinCancer kagglehub path: C:\Users\profe\.cache\kagglehub\datasets\fanconic\skin-cancer-malignant-vs-benign\versions\4
Contents: ['data', 'test', 'train']


  2%|▏         | 55.0M/2.29G [00:52<36:34, 1.10MB/s] 


KeyboardInterrupt: 

In [ ]:
# ===== Cell 3: Organize SkinCancer (fanconic) into data/SkinCancer/<class> =====
import pandas as pd  # used later for HAM10000 as well

skin_src_root = path_skin_simple
print("Using skin_src_root:", skin_src_root)
print("SkinCancer contents:", os.listdir(skin_src_root))

# Many versions of this dataset have 'train/' and 'test/' with benign/malignant.
# If the images are directly under benign/malignant, this still works.
dest_root_skin = os.path.join(DATA_ROOT, "SkinCancer")
os.makedirs(dest_root_skin, exist_ok=True)

splits = ["train", "test"]
copied = 0

for split in splits:
    split_dir = os.path.join(skin_src_root, split)
    if not os.path.exists(split_dir):
        continue

    for cls in os.listdir(split_dir):
        cls_src = os.path.join(split_dir, cls)
        if not os.path.isdir(cls_src):
            continue

        cls_dest = os.path.join(dest_root_skin, cls)
        os.makedirs(cls_dest, exist_ok=True)

        for fname in os.listdir(cls_src):
            src_path = os.path.join(cls_src, fname)
            if not os.path.isfile(src_path):
                continue
            dst_path = os.path.join(cls_dest, fname)
            shutil.copy(src_path, dst_path)
            copied += 1

print("SkinCancer organized into:", dest_root_skin)
print("Total images copied:", copied)
print("Final SkinCancer folders:", os.listdir(dest_root_skin))

In [ ]:
# ===== Cell 4: Organize HAM10000 into data/HAM10000/<class> =====

root_ham = os.path.join(DATA_ROOT, "HAM10000")
os.makedirs(root_ham, exist_ok=True)

metadata_path = os.path.join(path_ham, "HAM10000_metadata.csv")
df_ham = pd.read_csv(metadata_path)

ham_classes = df_ham["dx"].unique()
for c in ham_classes:
    os.makedirs(os.path.join(root_ham, c), exist_ok=True)

image_dirs = [
    os.path.join(path_ham, "HAM10000_images_part_1"),
    os.path.join(path_ham, "HAM10000_images_part_2"),
]
print("Image dirs to search:", image_dirs)

moved = 0
missing = 0

for img_id, dx in zip(df_ham["image_id"], df_ham["dx"]):
    fname = img_id + ".jpg"
    found = False
    for d in image_dirs:
        src = os.path.join(d, fname)
        if os.path.exists(src):
            dst = os.path.join(root_ham, dx, fname)
            shutil.copy(src, dst)
            moved += 1
            found = True
            break
    if not found:
        missing += 1

print("HAM10000 organized into:", root_ham)
print("Images copied:", moved, "| Missing:", missing)
print("HAM10000 class folders:", os.listdir(root_ham))

In [ ]:
# ===== Cell 5: Organize ChestXray into data/ChestXray/NORMAL,PNEUMONIA =====

print("ChestXray kagglehub path:", path_chest)
print("Contents at path_chest:", os.listdir(path_chest))

src_root_chest = path_chest
if "chest_xray" in os.listdir(path_chest):
    src_root_chest = os.path.join(path_chest, "chest_xray")

print("Using src_root_chest:", src_root_chest)

dest_root_cx = os.path.join(DATA_ROOT, "ChestXray")
os.makedirs(dest_root_cx, exist_ok=True)

for cls in ["NORMAL", "PNEUMONIA"]:
    os.makedirs(os.path.join(dest_root_cx, cls), exist_ok=True)

splits = ["train", "test", "val"]
copied_cx = 0

for split in splits:
    split_dir = os.path.join(src_root_chest, split)
    if not os.path.exists(split_dir):
        continue

    for cls in ["NORMAL", "PNEUMONIA"]:
        src_dir = os.path.join(split_dir, cls)
        if not os.path.exists(src_dir):
            continue

        dst_dir = os.path.join(dest_root_cx, cls)
        for fname in os.listdir(src_dir):
            src_path = os.path.join(src_dir, fname)
            if not os.path.isfile(src_path):
                continue
            dst_path = os.path.join(dst_dir, fname)
            shutil.copy(src_path, dst_path)
            copied_cx += 1

print("ChestXray organized into:", dest_root_cx)
print("Total ChestXray images copied:", copied_cx)
print("Final ChestXray folders:", os.listdir(dest_root_cx))

In [ ]:
# ===== Cell 6: Sanity check counts for all three datasets =====

def count_images(folder):
    exts = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.gif")
    total = 0
    for ext in exts:
        total += len(glob.glob(os.path.join(folder, ext)))
    return total

for ds in ["HAM10000", "SkinCancer", "ChestXray"]:
    root = os.path.join(DATA_ROOT, ds)
    print(f"\n=== {ds} ===")
    if not os.path.exists(root):
        print("Folder missing!")
        continue
    for cls in sorted(os.listdir(root)):
        cls_path = os.path.join(root, cls)
        if os.path.isdir(cls_path):
            print(cls, "->", count_images(cls_path), "images")

In [ ]:
# ===== Cell 7: EDA (no caching) on all datasets =====
from tensorflow import keras

DATASET_NAMES = ["HAM10000", "SkinCancer", "ChestXray"]
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

def get_datasets(dataset_name, validation_split=0.2):
    dataset_dir = os.path.join(DATA_ROOT, dataset_name)

    train_ds = keras.utils.image_dataset_from_directory(
        dataset_dir,
        labels="inferred",
        label_mode="categorical",
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=True,
        seed=SEED,
        validation_split=validation_split,
        subset="training",
    )

    val_ds = keras.utils.image_dataset_from_directory(
        dataset_dir,
        labels="inferred",
        label_mode="categorical",
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=True,
        seed=SEED,
        validation_split=validation_split,
        subset="validation",
    )

    class_names = train_ds.class_names
    num_classes = len(class_names)

    AUTOTUNE = tf.data.AUTOTUNE
    # IMPORTANT: no .cache() here to save RAM
    train_ds = train_ds.prefetch(AUTOTUNE)
    val_ds   = val_ds.prefetch(AUTOTUNE)

    return train_ds, val_ds, class_names, num_classes

def plot_class_distribution(train_ds, class_names, title=""):
    counts = np.zeros(len(class_names), dtype=int)
    for images, labels in train_ds:
        idx = np.argmax(labels.numpy(), axis=1)
        for i in idx:
            counts[i] += 1

    plt.figure(figsize=(8, 4))
    plt.bar(class_names, counts)
    plt.xticks(rotation=45)
    plt.title(title + " - class distribution")
    plt.tight_layout()
    plt.show()

def show_sample_images(train_ds, class_names, title="", num_images=9):
    plt.figure(figsize=(8, 8))
    for images, labels in train_ds.take(1):
        n = min(num_images, images.shape[0])
        for i in range(n):
            ax = plt.subplot(3, 3, i + 1)
            plt.imshow(images[i].numpy().astype("uint8"))
            lab = class_names[np.argmax(labels[i].numpy())]
            plt.title(lab, fontsize=8)
            plt.axis("off")
    plt.suptitle(title, y=0.95)
    plt.tight_layout()
    plt.show()

for ds in DATASET_NAMES:
    print("\n" + "="*60)
    print("Dataset:", ds)
    print("="*60)

    train_ds_tmp, val_ds_tmp, class_names_tmp, _ = get_datasets(ds)
    for images, labels in train_ds_tmp.take(1):
        print("Batch shape:", images.shape, labels.shape)
        break

    plot_class_distribution(train_ds_tmp, class_names_tmp, title=ds)
    show_sample_images(train_ds_tmp, class_names_tmp, title=ds)

In [ ]:
# ===== Cell 8: Preprocessing (augmentation + normalization) =====
from tensorflow.keras import layers

AUTOTUNE = tf.data.AUTOTUNE

data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.1),
        layers.RandomContrast(0.1),
    ],
    name="data_augmentation",
)

normalization = layers.Rescaling(1.0 / 255.0, name="rescale_0_1")

def preprocess_train(image, label, augment=True):
    if augment:
        image = data_augmentation(image)
    image = normalization(image)
    return image, label

def preprocess_val(image, label):
    image = normalization(image)
    return image, label

def get_preprocessed_datasets(dataset_name, validation_split=0.2, augment=True):
    raw_train, raw_val, class_names, num_classes = get_datasets(
        dataset_name, validation_split=validation_split
    )

    train_ds = raw_train.map(
        lambda x, y: preprocess_train(x, y, augment=augment),
        num_parallel_calls=AUTOTUNE,
    )
    val_ds = raw_val.map(
        preprocess_val,
        num_parallel_calls=AUTOTUNE,
    )

    train_ds = train_ds.prefetch(AUTOTUNE)
    val_ds   = val_ds.prefetch(AUTOTUNE)

    return train_ds, val_ds, class_names, num_classes

Now let us build the baseline CNN EfficientNetV2

In [ ]:
# ===== Cell 9: Build baseline CNN (EfficientNetV2) and training wrapper =====
from tensorflow.keras.applications import EfficientNetV2S

def build_efficientnetv2_baseline(input_shape, num_classes, train_base=False):
    inputs = keras.Input(shape=input_shape)
    base = EfficientNetV2S(
        include_top=False,
        weights="imagenet",
        input_tensor=inputs,
        pooling="avg",
    )
    base.trainable = train_base

    x = base.outputs[0]
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name="EffNetV2_baseline")
    return model

def compile_and_train(model, train_ds, val_ds, epochs=10, lr=1e-4):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
    )
    return history

In [ ]:
# ===== Cell 10: Metrics & evaluation helpers (fixed & simplified) =====
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
)

def compute_classification_metrics(y_true, y_prob):
    """
    y_true: 1D integer labels, shape [N]
    y_prob: predicted probabilities, shape [N, num_classes]
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    num_classes = y_prob.shape[1]

    y_pred = np.argmax(y_prob, axis=1)

    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    cm = confusion_matrix(y_true, y_pred, labels=range(num_classes))

    y_true_oh = tf.one_hot(y_true, depth=num_classes).numpy()
    auc_scores = []
    for c in range(num_classes):
        try:
            auc_c = roc_auc_score(y_true_oh[:, c], y_prob[:, c])
            auc_scores.append(auc_c)
        except ValueError:
            pass
    auc = float(np.mean(auc_scores)) if len(auc_scores) > 0 else float("nan")

    return {
        "Acc": float(acc),
        "Precision": float(precision),
        "Recall": float(recall),
        "F1": float(f1),
        "AUC": auc,
        "confusion_matrix": cm,
    }

def evaluate_model_on_dataset(model, dataset, class_names, max_batches=None):
    all_probs = []
    all_labels = []

    for i, (images, labels) in enumerate(dataset):
        probs = model.predict(images, verbose=0)
        label_ids = np.argmax(labels.numpy(), axis=1)

        all_probs.append(probs)
        all_labels.append(label_ids)

        if max_batches is not None and (i + 1) >= max_batches:
            break

    all_probs = np.concatenate(all_probs, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    assert all_probs.shape[0] == all_labels.shape[0], (
        f"Mismatch: probs {all_probs.shape[0]} vs labels {all_labels.shape[0]}"
    )

    metrics = compute_classification_metrics(all_labels, all_probs)

    print("\nEvaluation:")
    for key in ["Acc", "Precision", "Recall", "F1", "AUC"]:
        val = metrics[key]
        if isinstance(val, float) and not np.isnan(val):
            print(f"{key}: {val:.4f}")
        else:
            print(f"{key}: {val}")
    print("\nConfusion matrix:")
    print(metrics["confusion_matrix"])

    return metrics

In [ ]:
# ===== Cell 11: Initialize results container =====
results = []  # list of dicts: {"Dataset", "Model", "Acc", "F1", "AUC", ...}

In [ ]:
# ===== Cell 12: Train + evaluate baseline on SkinCancer (sequential) =====
input_shape = IMAGE_SIZE + (3,)

skin_train, skin_val, skin_classes, skin_n = get_preprocessed_datasets("SkinCancer")

baseline_skin = build_efficientnetv2_baseline(input_shape, skin_n, train_base=False)
baseline_skin.summary()

history_skin = compile_and_train(
    baseline_skin,
    skin_train,
    skin_val,
    epochs=10,    # keep same as before
    lr=1e-4,
)

metrics_baseline_skin = evaluate_model_on_dataset(
    baseline_skin,
    skin_val,
    skin_classes,
)

results.append({
    "Dataset": "SkinCancer",
    "Model": "EffNetV2-baseline",
    "Acc":  metrics_baseline_skin["Acc"],
    "F1":   metrics_baseline_skin["F1"],
    "AUC":  metrics_baseline_skin["AUC"],
})

# free memory before next dataset
del skin_train, skin_val, baseline_skin, history_skin
tf.keras.backend.clear_session()

In [ ]:
# ===== Cell 13: Train + evaluate baseline on HAM10000 (sequential) =====
input_shape = IMAGE_SIZE + (3,)

ham_train, ham_val, ham_classes, ham_n = get_preprocessed_datasets("HAM10000")

baseline_ham = build_efficientnetv2_baseline(input_shape, ham_n, train_base=False)
baseline_ham.summary()

history_ham = compile_and_train(
    baseline_ham,
    ham_train,
    ham_val,
    epochs=10,
    lr=1e-4,
)

metrics_baseline_ham = evaluate_model_on_dataset(
    baseline_ham,
    ham_val,
    ham_classes,
)

results.append({
    "Dataset": "HAM10000",
    "Model": "EffNetV2-baseline",
    "Acc":  metrics_baseline_ham["Acc"],
    "F1":   metrics_baseline_ham["F1"],
    "AUC":  metrics_baseline_ham["AUC"],
})

del ham_train, ham_val, baseline_ham, history_ham
tf.keras.backend.clear_session()

In [ ]:
# ===== Cell 14: Train + evaluate baseline on ChestXray (sequential) =====
input_shape = IMAGE_SIZE + (3,)

cx_train, cx_val, cx_classes, cx_n = get_preprocessed_datasets("ChestXray")

baseline_cx = build_efficientnetv2_baseline(input_shape, cx_n, train_base=False)
baseline_cx.summary()

history_cx = compile_and_train(
    baseline_cx,
    cx_train,
    cx_val,
    epochs=10,
    lr=1e-4,
)

metrics_baseline_cx = evaluate_model_on_dataset(
    baseline_cx,
    cx_val,
    cx_classes,
)

results.append({
    "Dataset": "ChestXray",
    "Model": "EffNetV2-baseline",
    "Acc":  metrics_baseline_cx["Acc"],
    "F1":   metrics_baseline_cx["F1"],
    "AUC":  metrics_baseline_cx["AUC"],
})

del cx_train, cx_val, baseline_cx, history_cx
tf.keras.backend.clear_session()

In [ ]:
# ===== Cell 15: Summary results table =====
import pandas as pd

df_results = pd.DataFrame(results)
display(df_results)

# Optional pivot tables for quick comparison
acc_table = df_results.pivot(index="Dataset", columns="Model", values="Acc")
f1_table  = df_results.pivot(index="Dataset", columns="Model", values="F1")
auc_table = df_results.pivot(index="Dataset", columns="Model", values="AUC")

print("\nAccuracy table:")
display(acc_table)

print("\nF1 table:")
display(f1_table)

print("\nAUC table:")
display(auc_table)

# Save to CSV (later plotting)
df_results.to_csv("results_baseline.csv", index=False)
print("\nSaved results to results_baseline.csv")

Now let us work on EffPCNet-small

In [ ]:
# ===== Cell 16: EffPCNet-small building blocks =====
from tensorflow.keras import layers

def se_block(x, reduction=4, name=None):
    """Simple Squeeze-and-Excitation block."""
    filters = x.shape[-1]
    se = layers.GlobalAveragePooling2D(name=f"{name}_gap" if name else None)(x)
    se = layers.Reshape((1, 1, filters))(se)
    se = layers.Dense(filters // reduction, activation="swish",
                      name=f"{name}_fc1" if name else None)(se)
    se = layers.Dense(filters, activation="sigmoid",
                      name=f"{name}_fc2" if name else None)(se)
    return layers.Multiply(name=f"{name}_scale" if name else None)([x, se])

def conv_bn_act(x, filters, kernel_size, stride=1, name=None):
    x = layers.Conv2D(filters, kernel_size, strides=stride, padding="same",
                      name=f"{name}_conv" if name else None)(x)
    x = layers.BatchNormalization(name=f"{name}_bn" if name else None)(x)
    x = layers.Activation("swish", name=f"{name}_act" if name else None)(x)
    return x

def m2c_block(x, name=None):
    """
    M2C-like block: split channels into 4 branches with different depthwise convs
    and then fuse. This is a lightweight version inspired by the paper.
    """
    channels = x.shape[-1]
    c_small = channels // 8
    c_large = channels - 3 * c_small

    x1, x2, x3, x4 = tf.split(x, [c_small, c_small, c_small, c_large], axis=-1)

    # 3x3 depthwise
    b1 = layers.DepthwiseConv2D(3, padding="same", name=f"{name}_dw3x3")(x1)
    b1 = layers.BatchNormalization(name=f"{name}_bn1")(b1)
    b1 = layers.ReLU(name=f"{name}_relu1")(b1)

    # 1xK then Kx1 approximating large kernels
    K = 11
    b2 = layers.DepthwiseConv2D((1, K), padding="same", name=f"{name}_dw1xK")(x2)
    b2 = layers.BatchNormalization(name=f"{name}_bn2")(b2)
    b2 = layers.ReLU(name=f"{name}_relu2")(b2)

    b3 = layers.DepthwiseConv2D((K, 1), padding="same", name=f"{name}_dwKx1")(x3)
    b3 = layers.BatchNormalization(name=f"{name}_bn3")(b3)
    b3 = layers.ReLU(name=f"{name}_relu3")(b3)

    # Identity branch
    b4 = x4

    out = layers.Concatenate(axis=-1, name=f"{name}_concat")([b1, b2, b3, b4])
    out = layers.Conv2D(channels, 1, padding="same", name=f"{name}_conv1x1")(out)
    out = layers.BatchNormalization(name=f"{name}_bn_out")(out)
    out = layers.ReLU(name=f"{name}_relu_out")(out)
    return out

def mbc_block(x, filters, stride=1, name=None, repeats=1):
    """
    Simplified MBC-style residual block: conv -> conv -> SE -> residual add.
    """
    for i in range(repeats):
        shortcut = x
        if stride != 1 or x.shape[-1] != filters:
            shortcut = layers.Conv2D(filters, 1, strides=stride, padding="same",
                                     name=f"{name}_proj{i}")(x)
            shortcut = layers.BatchNormalization(name=f"{name}_proj_bn{i}")(shortcut)

        out = conv_bn_act(x, filters, 3, stride=stride, name=f"{name}_c1_{i}")
        out = conv_bn_act(out, filters, 3, stride=1, name=f"{name}_c2_{i}")
        out = se_block(out, reduction=4, name=f"{name}_se_{i}")
        out = layers.Add(name=f"{name}_add_{i}")([shortcut, out])
        out = layers.Activation("swish", name=f"{name}_out_{i}")(out)

        x = out
        stride = 1  # only first repeat uses stride
    return x

def rep_c_block(x, filters, stride=1, name=None):
    """
    Simplified Rep-C-style block: multiple conv branches + SE + residual.
    """
    shortcut = x
    if stride != 1 or x.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, padding="same",
                                 name=f"{name}_shortcut_conv")(x)
        shortcut = layers.BatchNormalization(name=f"{name}_shortcut_bn")(shortcut)

    # 3x3 conv branch
    b1 = layers.Conv2D(filters, 3, strides=stride, padding="same",
                       name=f"{name}_conv3x3")(x)
    b1 = layers.BatchNormalization(name=f"{name}_bn3x3")(b1)
    b1 = layers.ReLU(name=f"{name}_relu3x3")(b1)

    # 1x1 conv branch
    b2 = layers.Conv2D(filters, 1, strides=stride, padding="same",
                       name=f"{name}_conv1x1")(x)
    b2 = layers.BatchNormalization(name=f"{name}_bn1x1")(b2)
    b2 = layers.ReLU(name=f"{name}_relu1x1")(b2)

    # depthwise branch
    b3 = layers.DepthwiseConv2D(3, strides=stride, padding="same",
                                name=f"{name}_dw")(x)
    b3 = layers.BatchNormalization(name=f"{name}_bndw")(b3)
    b3 = layers.ReLU(name=f"{name}_reludw")(b3)

    out = layers.Add(name=f"{name}_add")([shortcut, b1, b2, b3])
    out = se_block(out, reduction=4, name=f"{name}_se")
    out = layers.ReLU(name=f"{name}_relu_out")(out)
    return out

def build_effpcnet_small(input_shape, num_classes):
    """
    EffPCNet-inspired small network:
    Stem -> basic conv -> M2C -> MBC -> Rep-C -> deeper MBC -> head.
    """
    inputs = keras.Input(shape=input_shape)

    # Stem
    x = conv_bn_act(inputs, 24, 3, stride=2, name="stem")
    x = conv_bn_act(x, 32, 3, stride=2, name="stage1")
    x = conv_bn_act(x, 48, 3, stride=2, name="stage2")
    x = conv_bn_act(x, 64, 3, stride=1, name="stage3")

    # Stage 4: M2C
    x = m2c_block(x, name="stage4_m2c")

    # Stage 5: MBC
    x = mbc_block(x, 128, stride=2, name="stage5_mbc", repeats=2)

    # Stage 6: Rep-C
    x = rep_c_block(x, 128, stride=1, name="stage6_rep")

    # Stage 7–8: deeper MBC
    x = mbc_block(x, 192, stride=2, name="stage7_mbc", repeats=2)
    x = mbc_block(x, 256, stride=2, name="stage8_mbc", repeats=2)

    # Head
    x = layers.GlobalAveragePooling2D(name="global_pool")(x)
    x = layers.Dropout(0.3, name="dropout")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="pred")(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name="EffPCNet_small")
    return model

In [ ]:
# ===== Cell 17: EffPCNet-small on SkinCancer =====
input_shape = IMAGE_SIZE + (3,)

skin_train, skin_val, skin_classes, skin_n = get_preprocessed_datasets("SkinCancer")

effpcnet_skin = build_effpcnet_small(input_shape, skin_n)
effpcnet_skin.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

effpcnet_skin.summary()

history_effpcnet_skin = effpcnet_skin.fit(
    skin_train,
    validation_data=skin_val,
    epochs=10,           # same as baseline
)

metrics_effpcnet_skin = evaluate_model_on_dataset(
    effpcnet_skin,
    skin_val,
    skin_classes,
)

results.append({
    "Dataset": "SkinCancer",
    "Model": "EffPCNet-small",
    "Acc":  metrics_effpcnet_skin["Acc"],
    "F1":   metrics_effpcnet_skin["F1"],
    "AUC":  metrics_effpcnet_skin["AUC"],
})

del skin_train, skin_val, effpcnet_skin, history_effpcnet_skin
tf.keras.backend.clear_session()

In [ ]:
# ===== Cell 18: EffPCNet-small on HAM10000 =====
input_shape = IMAGE_SIZE + (3,)

ham_train, ham_val, ham_classes, ham_n = get_preprocessed_datasets("HAM10000")

effpcnet_ham = build_effpcnet_small(input_shape, ham_n)
effpcnet_ham.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

effpcnet_ham.summary()

history_effpcnet_ham = effpcnet_ham.fit(
    ham_train,
    validation_data=ham_val,
    epochs=10,
)

metrics_effpcnet_ham = evaluate_model_on_dataset(
    effpcnet_ham,
    ham_val,
    ham_classes,
)

results.append({
    "Dataset": "HAM10000",
    "Model": "EffPCNet-small",
    "Acc":  metrics_effpcnet_ham["Acc"],
    "F1":   metrics_effpcnet_ham["F1"],
    "AUC":  metrics_effpcnet_ham["AUC"],
})

del ham_train, ham_val, effpcnet_ham, history_effpcnet_ham
tf.keras.backend.clear_session()

In [ ]:
# ===== Cell 19: EffPCNet-small on ChestXray =====
input_shape = IMAGE_SIZE + (3,)

cx_train, cx_val, cx_classes, cx_n = get_preprocessed_datasets("ChestXray")

effpcnet_cx = build_effpcnet_small(input_shape, cx_n)
effpcnet_cx.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

effpcnet_cx.summary()

history_effpcnet_cx = effpcnet_cx.fit(
    cx_train,
    validation_data=cx_val,
    epochs=10,
)

metrics_effpcnet_cx = evaluate_model_on_dataset(
    effpcnet_cx,
    cx_val,
    cx_classes,
)

results.append({
    "Dataset": "ChestXray",
    "Model": "EffPCNet-small",
    "Acc":  metrics_effpcnet_cx["Acc"],
    "F1":   metrics_effpcnet_cx["F1"],
    "AUC":  metrics_effpcnet_cx["AUC"],
})

del cx_train, cx_val, effpcnet_cx, history_effpcnet_cx
tf.keras.backend.clear_session()

In [ ]:
# ===== Cell 20: Final summary for report =====
import pandas as pd

df_results = pd.DataFrame(results)
display(df_results)

acc_table = df_results.pivot(index="Dataset", columns="Model", values="Acc")
f1_table  = df_results.pivot(index="Dataset", columns="Model", values="F1")
auc_table = df_results.pivot(index="Dataset", columns="Model", values="AUC")

print("\nAccuracy table:")
display(acc_table)

print("\nF1 table:")
display(f1_table)

print("\nAUC table:")
display(auc_table)

df_results.to_csv("results_all_models.csv", index=False)
print("\nSaved results to results_all_models.csv")